<a href="https://colab.research.google.com/github/mhowlin-web/TP_RAG_ARCA/blob/main/03_chunking_corpus_arca.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP RAG y Agentes

## Asistente para consultas sobre trámites de Monotributo en ARCA

En este notebook voy a dividir los documentos del corpus de ARCA en
fragmentos más pequeños, llamados chunks.

El objetivo es preparar los documentos para la generación de embeddings.

No quiero generar un embedding para una página web completa, porque los
documentos pueden contener demasiada información y una consulta concreta
podría quedar mezclada con contenido que no es relevante.

Por eso divido cada documento en fragmentos relativamente pequeños.

En esta etapa todavía no utilizo Pinecone ni genero embeddings.

El resultado de este notebook será un nuevo archivo JSON que contiene
todos los chunks junto con los metadatos de los documentos originales.

## Montaje de Google Drive

En esta celda monto Google Drive para acceder al corpus limpio que generé
en el Notebook 02.

De esta manera todos los notebooks utilizan la misma ubicación persistente
para los archivos del proyecto.

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


## Definición de la carpeta del proyecto

En esta celda defino la carpeta de Google Drive donde estoy almacenando
el proyecto.

Utilizo la misma ruta que en los notebooks anteriores para evitar
problemas al compartir archivos entre notebooks.

In [2]:
from pathlib import Path

CARPETA_PROYECTO = Path(
    "/content/drive/MyDrive/TP_RAG_ARCA"
)

CARPETA_CORPUS = (
    CARPETA_PROYECTO / "corpus"
)

print("Carpeta del proyecto:")
print(CARPETA_PROYECTO)

print("\nCarpeta del corpus:")
print(CARPETA_CORPUS)

Carpeta del proyecto:
/content/drive/MyDrive/TP_RAG_ARCA

Carpeta del corpus:
/content/drive/MyDrive/TP_RAG_ARCA/corpus


## Definición de los archivos

En esta celda defino el archivo de entrada y el archivo de salida.

El archivo de entrada es el corpus limpio generado en el Notebook 02.

El archivo de salida contendrá los chunks que voy a utilizar posteriormente
para generar embeddings.

In [3]:
ARCHIVO_ENTRADA = (
    CARPETA_CORPUS /
    "corpus_arca_monotributo_limpio.json"
)

ARCHIVO_SALIDA = (
    CARPETA_CORPUS /
    "corpus_arca_monotributo_chunks.json"
)

print("Archivo de entrada:")
print(ARCHIVO_ENTRADA)

print("\nArchivo de salida:")
print(ARCHIVO_SALIDA)

Archivo de entrada:
/content/drive/MyDrive/TP_RAG_ARCA/corpus/corpus_arca_monotributo_limpio.json

Archivo de salida:
/content/drive/MyDrive/TP_RAG_ARCA/corpus/corpus_arca_monotributo_chunks.json


## Verificación del archivo de entrada

En esta celda verifico que el corpus limpio generado en el Notebook 02
exista antes de comenzar el procesamiento.

In [4]:
if not ARCHIVO_ENTRADA.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo:\n{ARCHIVO_ENTRADA}"
    )

print("Archivo encontrado correctamente.")
print(
    f"Tamaño: {ARCHIVO_ENTRADA.stat().st_size:,} bytes"
)

Archivo encontrado correctamente.
Tamaño: 27,484 bytes


## Importación de librerías

En esta celda importo las librerías que necesito.

Utilizo `json` para leer y guardar el corpus y `re` para realizar una
pequeña normalización de los espacios.

In [5]:
import json
import re

## Carga del corpus limpio

En esta celda cargo el corpus limpio generado anteriormente.

Cada elemento del corpus representa un documento de ARCA y contiene
su texto y sus metadatos.

In [6]:
with open(
    ARCHIVO_ENTRADA,
    "r",
    encoding="utf-8"
) as archivo:

    corpus = json.load(archivo)

print(
    f"Documentos cargados: {len(corpus)}"
)

Documentos cargados: 8


## Inspección del corpus

En esta celda reviso los documentos antes de dividirlos.

Muestro el título y la cantidad de caracteres de cada documento para
tener una idea del tamaño del corpus.

In [7]:
print("RESUMEN DEL CORPUS")
print("=" * 80)

for documento in corpus:

    print(
        f"\nID: {documento['id']}"
    )

    print(
        f"Título: {documento['titulo']}"
    )

    print(
        f"Caracteres: {len(documento['texto'])}"
    )

RESUMEN DEL CORPUS

ID: inicio
Título: Inicio - Ayuda sobre el Monotributo
Caracteres: 1967

ID: clave_fiscal
Título: Obtención de Clave Fiscal
Caracteres: 2615

ID: constancias
Título: Constancias y credenciales
Caracteres: 2578

ID: facturacion
Título: Facturación
Caracteres: 4181

ID: recategorizacion
Título: Recategorización
Caracteres: 4583

ID: baja
Título: Baja de monotributo
Caracteres: 2253

ID: desarrollo_actividad
Título: Desarrollo de la actividad
Caracteres: 2171

ID: tutoriales
Título: Tutoriales sobre Monotributo
Caracteres: 3904


## Parámetros del chunking

En esta celda defino el tamaño de los chunks y el solapamiento.

Utilizo un tamaño de 800 caracteres y un solapamiento de 150 caracteres.

El solapamiento permite que una información que se encuentra cerca del
límite entre dos chunks no quede completamente separada.

Estos valores son parámetros iniciales. Más adelante puedo experimentar
con otros tamaños para evaluar cómo afectan al retrieval.

In [8]:
TAMANO_CHUNK = 800
OVERLAP = 150

if OVERLAP >= TAMANO_CHUNK:
    raise ValueError(
        "OVERLAP debe ser menor que TAMANO_CHUNK."
    )

print(f"Tamaño del chunk: {TAMANO_CHUNK}")
print(f"Solapamiento: {OVERLAP}")

Tamaño del chunk: 800
Solapamiento: 150


## Función de limpieza previa al chunking

En esta celda creo una función sencilla para normalizar espacios.

No vuelvo a corregir el encoding porque esa tarea ya fue realizada
en el Notebook 02.

Acá solamente preparo el texto para dividirlo en fragmentos.

In [9]:
def normalizar_texto(texto):

    texto = re.sub(
        r"[ \t]+",
        " ",
        texto
    )

    texto = re.sub(
        r"\n\s*\n+",
        "\n\n",
        texto
    )

    return texto.strip()

## Función para dividir un documento en chunks

En esta celda implemento el algoritmo de chunking.

Recorro el texto utilizando una ventana de tamaño fijo.

Después de generar un chunk avanzo una cantidad menor que el tamaño
total del chunk. La diferencia genera el solapamiento.

Por ejemplo, si el tamaño es 800 y el solapamiento es 150, el siguiente
chunk comienza 650 caracteres después del comienzo del anterior.

In [10]:
def dividir_en_chunks(
    texto,
    tamano_chunk=TAMANO_CHUNK,
    overlap=OVERLAP
):

    texto = normalizar_texto(texto)

    if not texto:
        return []

    chunks = []

    paso = tamano_chunk - overlap

    inicio = 0

    while inicio < len(texto):

        fin = inicio + tamano_chunk

        chunk = texto[inicio:fin].strip()

        if chunk:
            chunks.append(chunk)

        inicio += paso

    return chunks

## Prueba del chunking

En esta celda pruebo la función sobre el primer documento.

Muestro la cantidad de chunks generados y algunos ejemplos.

Antes de procesar todo el corpus quiero verificar que el procedimiento
funcione correctamente.

In [11]:
documento_prueba = corpus[0]

chunks_prueba = dividir_en_chunks(
    documento_prueba["texto"]
)

print(
    f"Documento: {documento_prueba['titulo']}"
)

print(
    f"Caracteres: {len(documento_prueba['texto'])}"
)

print(
    f"Chunks generados: {len(chunks_prueba)}"
)

Documento: Inicio - Ayuda sobre el Monotributo
Caracteres: 1967
Chunks generados: 4


## Visualización de los primeros chunks

En esta celda inspecciono algunos de los chunks generados.

Quiero comprobar que cada fragmento tenga suficiente contexto y que
el texto no haya sido alterado durante la división.

In [12]:
for i, chunk in enumerate(
    chunks_prueba[:5],
    start=1
):

    print("=" * 80)
    print(f"CHUNK {i}")
    print("=" * 80)

    print(chunk)

    print(
        f"\nCaracteres: {len(chunk)}"
    )

CHUNK 1
Inicio - Ayuda sobre el monotributo - Monotributo | ARCA
Evitar las herramientas de navegación y pasar al contenido
Monotributo
Menu
Inicio
Ayuda
Inicio
Ayuda sobre el monotributo
Inicio
Ayuda sobre el monotributo
Toda la información sobre cómo darte de alta y hacer operaciones como monotributista.
Ingresar con clave fiscal
Menú de contenidos
Qué es
INSCRIPCIÓN
Inicio
Clave fiscal
CUIT
Domicilio Fiscal Electrónico
Jurisdicciones
Actividades
ALTA DE MONOTRIBUTO
Procedimiento
Tipos de monotributo
Parámetros
Jubilación
Obra social
Monotributo unificado
Constancias y credenciales
DESPUÉS DEL ALTA
Desarrollo de la actividad
Facturación
Pagos
Recategorización
FINALIZACIÓN DE ACTIVIDADES
Baja
Por cese de actividades
De oficio
Exclusión
Renuncia
Pasaje al régimen general
Ayuda
Inicio
El primer pas

Caracteres: 800
CHUNK 2
egorización
FINALIZACIÓN DE ACTIVIDADES
Baja
Por cese de actividades
De oficio
Exclusión
Renuncia
Pasaje al régimen general
Ayuda
Inicio
El primer paso es inscribirse

## Creación del corpus de chunks

En esta celda aplico el chunking a todos los documentos del corpus.

Además de conservar el texto del chunk, guardo los metadatos del documento
original.

Cada chunk tendrá:

- Un identificador propio.
- El identificador del documento original.
- El título.
- El organismo.
- La URL.
- El número de chunk.
- El texto del chunk.

Estos metadatos serán importantes posteriormente para Pinecone y para
poder identificar la fuente de una respuesta del RAG.

In [13]:
chunks_corpus = []

for documento in corpus:

    chunks = dividir_en_chunks(
        documento["texto"]
    )

    for numero_chunk, texto_chunk in enumerate(
        chunks,
        start=0
    ):

        chunk = {
            "chunk_id": (
                f"{documento['id']}_{numero_chunk}"
            ),
            "documento_id": documento["id"],
            "titulo": documento["titulo"],
            "organismo": documento["organismo"],
            "url": documento["url"],
            "fecha_descarga": documento["fecha_descarga"],
            "chunk": numero_chunk,
            "texto": texto_chunk
        }

        chunks_corpus.append(chunk)

print(
    f"Chunks generados: {len(chunks_corpus)}"
)

Chunks generados: 43


## Resumen del chunking

En esta celda resumo cuántos chunks produjo cada documento.

Esto me permite detectar si algún documento produjo una cantidad
inusualmente pequeña o grande de fragmentos.

In [14]:
print("RESUMEN DEL CHUNKING")
print("=" * 80)

for documento in corpus:

    cantidad = sum(
        1
        for chunk in chunks_corpus
        if chunk["documento_id"] == documento["id"]
    )

    print(
        f"{documento['id']}: "
        f"{cantidad} chunks"
    )

RESUMEN DEL CHUNKING
inicio: 4 chunks
clave_fiscal: 5 chunks
constancias: 4 chunks
facturacion: 7 chunks
recategorizacion: 8 chunks
baja: 4 chunks
desarrollo_actividad: 4 chunks
tutoriales: 7 chunks


## Estadísticas de los chunks

En esta celda calculo algunas estadísticas simples sobre los tamaños
de los chunks.

Esto me permite verificar que el chunking produjo fragmentos con
tamaños razonables.

In [15]:
tamanos = [
    len(chunk["texto"])
    for chunk in chunks_corpus
]

print(
    f"Cantidad de chunks: {len(tamanos)}"
)

print(
    f"Tamaño mínimo: {min(tamanos)}"
)

print(
    f"Tamaño máximo: {max(tamanos)}"
)

print(
    f"Tamaño promedio: "
    f"{sum(tamanos) / len(tamanos):.2f}"
)

Cantidad de chunks: 43
Tamaño mínimo: 4
Tamaño máximo: 800
Tamaño promedio: 673.37


## Inspección de un chunk completo

En esta celda selecciono un chunk y muestro tanto sus metadatos como
su contenido.

Esta estructura será muy similar a la que posteriormente enviaré
a Pinecone junto con su embedding.

In [16]:
chunk = chunks_corpus[0]

print("=" * 80)
print("METADATOS")
print("=" * 80)

for clave, valor in chunk.items():

    if clave != "texto":
        print(
            f"{clave}: {valor}"
        )

print("\n")
print("=" * 80)
print("TEXTO")
print("=" * 80)

print(chunk["texto"])

METADATOS
chunk_id: inicio_0
documento_id: inicio
titulo: Inicio - Ayuda sobre el Monotributo
organismo: ARCA
url: https://www.arca.gob.ar/monotributo/ayuda/inicio.asp
fecha_descarga: 2026-08-30T00:00:04.526776+00:00
chunk: 0


TEXTO
Inicio - Ayuda sobre el monotributo - Monotributo | ARCA
Evitar las herramientas de navegación y pasar al contenido
Monotributo
Menu
Inicio
Ayuda
Inicio
Ayuda sobre el monotributo
Inicio
Ayuda sobre el monotributo
Toda la información sobre cómo darte de alta y hacer operaciones como monotributista.
Ingresar con clave fiscal
Menú de contenidos
Qué es
INSCRIPCIÓN
Inicio
Clave fiscal
CUIT
Domicilio Fiscal Electrónico
Jurisdicciones
Actividades
ALTA DE MONOTRIBUTO
Procedimiento
Tipos de monotributo
Parámetros
Jubilación
Obra social
Monotributo unificado
Constancias y credenciales
DESPUÉS DEL ALTA
Desarrollo de la actividad
Facturación
Pagos
Recategorización
FINALIZACIÓN DE ACTIVIDADES
Baja
Por cese de actividades
De oficio
Exclusión
Renuncia
Pasaje al régimen 

## Guardado del corpus de chunks

En esta celda guardo los chunks en formato JSON.

Este archivo será la entrada del siguiente notebook, donde voy a
generar embeddings utilizando un modelo de Hugging Face.

Todavía no envío nada a Pinecone.

In [17]:
with open(
    ARCHIVO_SALIDA,
    "w",
    encoding="utf-8"
) as archivo:

    json.dump(
        chunks_corpus,
        archivo,
        ensure_ascii=False,
        indent=2
    )

print(
    "Archivo guardado correctamente:"
)

print(
    ARCHIVO_SALIDA
)

Archivo guardado correctamente:
/content/drive/MyDrive/TP_RAG_ARCA/corpus/corpus_arca_monotributo_chunks.json


## Verificación del archivo de salida

En esta celda vuelvo a cargar el archivo generado para comprobar
que fue guardado correctamente.

También verifico que la cantidad de chunks coincida con la cantidad
que tenía en memoria.

In [18]:
with open(
    ARCHIVO_SALIDA,
    "r",
    encoding="utf-8"
) as archivo:

    chunks_verificados = json.load(
        archivo
    )

print(
    f"Chunks en memoria: {len(chunks_corpus)}"
)

print(
    f"Chunks en archivo: {len(chunks_verificados)}"
)

if len(chunks_corpus) != len(chunks_verificados):

    raise ValueError(
        "La cantidad de chunks no coincide."
    )

print("\nVerificación OK.")

Chunks en memoria: 43
Chunks en archivo: 43

Verificación OK.


## Conclusión

En este notebook dividí el corpus limpio de ARCA en chunks de tamaño
controlado.

Utilicé un tamaño de 800 caracteres y un solapamiento de 150 caracteres.

Cada chunk conserva los metadatos necesarios para identificar su
documento de origen.

El resultado se guarda en:

    corpus_arca_monotributo_chunks.json

En el próximo notebook voy a generar un embedding para cada chunk.

Estos embeddings serán posteriormente almacenados en Pinecone para
poder realizar búsquedas semánticas sobre el corpus.